# Model V
### Audio Spectrogram Transformer - trening sieci

Biblioteki

In [1]:
import sys
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from model import InstrumentASTPretrained
from trening import get_device, train_model
from przygotowanie_danych import INSTRUMENTS, create_datasets, create_loaders


/opt/anaconda3/envs/cnn/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1126 11:27:12.241000 27178 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


### Konfiguracja eksperymentu

In [2]:
# Identyfikator przygotowanego zbioru danych.
dataset = "1_3"
model_name = "AST"

epochs = 100
patience = 10
batch_size = 8
learning_rate = 0.0001

dropout = 0.1
threshold = 0.8

hidden_size = 768
num_hidden_layers = 12
num_attention_heads = 12
intermediate_size = 3072

results_file = Path("wyniki/modele.csv")


### Sprawdzenie, czy eksperyment był już wykonywany

In [3]:
if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout"] == dropout) &
        (results["Hidden size"] == hidden_size) &
        (results["Transformer layers"] == num_hidden_layers) &
        (results["Attention heads"] == num_attention_heads) &
        (results["Intermediate size"] == intermediate_size)
    ).any()

    if combination_exists:
        raise ValueError(
            "Ta kombinacja datasetu i hiperparametrów "
            "była już testowana. Zatrzymuję program."
        )

print("Nowa kombinacja. Kontynuuję wykonywanie kodu.")

Nowa kombinacja. Kontynuuję wykonywanie kodu.


### Nadanie identyfikatora eksperymentu

In [4]:
results_dir = Path("wyniki")
results_dir.mkdir(parents=True, exist_ok=True)

existing_numbers = [
    int(f.stem)
    for f in results_dir.glob("*.pth")
    if f.stem.isdigit()
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

print(f"Learning ID: {learning_id}")
print(f"Plik wag: {weights_filename}")

Learning ID: 5
Plik wag: 5.pth


### Przygotowanie danych

In [5]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(train_dataset, val_dataset, test_dataset, batch_size=batch_size)

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

# Pobranie przykładowego spektrogramu ze zbioru treningowego.
sample_X, sample_y = train_dataset[0]

# Automatyczne określenie wymiarów wejściowych AST.
num_mel_bins = sample_X.shape[-2]
max_length = sample_X.shape[-1]

print(f"Liczba pasm mel: {num_mel_bins}")
print(f"Długość sekwencji: {max_length}")

Wczytuję spektrogramy z: /Users/khaczkiewicz/Documents/projekt-instrumenty/instrument-recognition-project/spectrograms/1_3
Train: 7069
Validation: 1905
Test: 1407
Liczba pasm mel: 128
Długość sekwencji: 346


### Inicjalizacja modelu

In [6]:
device = get_device()
print(f"Urządzenie: {device}")

model = InstrumentASTPretrained(dropout=dropout).to(device)

criterion = nn.MultiLabelSoftMarginLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Urządzenie: mps


[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `527`.
Loading weights: 100%|██████████| 203/203 [00:00<00:00, 23539.40it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                         
------------------------+----------+-----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([7, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([7])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


### Trening sieci

In [ ]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name=weights_filename
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

/opt/anaconda3/envs/cnn/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


### Zapis informacji o eksperymencie

In [ ]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout": dropout,
    "Hidden size": hidden_size,
    "Transformer layers": num_hidden_layers,
    "Attention heads": num_attention_heads,
    "Intermediate size": intermediate_size,
    "Threshold": threshold,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    results_dir / f"history_{learning_id}.csv",
    index_label="epoch"
)

print(f"Informacje o eksperymencie zapisano w: {results_file}")
print(f"Historia uczenia: history_{learning_id}.csv")

Informacje o eksperymencie zapisano w: wyniki/modele.csv
Historia uczenia: history_4.csv
